# BCM3553 — Cours 2
### Mettre la logique en fonction et l’appliquer à de vraies données

Ce notebook accompagne la démonstration en direct du cours. Chaque cellule contient exactement le code présenté sur les diapositives — exécutez-les dans l’ordre avec **Maj+Entrée**.

**Fichier de données utilisé aujourd’hui :** `cttn_teaching_multiset.fa` — le même fichier FASTA pédagogique de 5 protéines utilisé pendant l’heure Unix du cours 1. Ce fichier se trouve ici: `../z.data/data_seq/cttn_teaching_multiset.fa`.


## Dernière heure : décisions et répétition

`if / elif / else`, l’opérateur `in`, les boucles `for` et le compteur manuel qui a classé 10 résidus à la main : 2 chargés, 1 polaire, 7 hydrophobes. Aujourd’hui : emballer cette logique dans des fonctions réutilisables et l’appliquer à un vrai fichier.


## Pourquoi des fonctions ?

Votre classificateur fonctionne — mais seulement sur un extrait saisi à la main. Copier-coller cette boucle pour chaque nouvelle protéine est exactement le genre de répétition dont Python devrait vous épargner. Une fonction vous permet d’écrire la logique une fois, de lui donner un nom et de l’appeler autant de fois que nécessaire.


## `def` — définir une fonction


In [1]:
def greet():
    print("Bonjour, labo!")

greet()

Bonjour, labo!


`def` la définit — rien ne s’exécute encore. C’est `greet()` qui l’appelle réellement.


## Paramètres — passer des valeurs


In [3]:
def greet(name):
    print("Bonjour,", name)

greet("Foisy")

Bonjour, Foisy


`name` est un paramètre — un emplacement rempli par ce que vous transmettez lorsque vous appelez la fonction.


**À essayer :** une fonction avec deux paramètres.


In [1]:
#
# Noter que si vous avez besoin de deux paramètres, en oublier un
# vous donnera une erreur...
#
# Diverses façons existent pour tourner autour du problème mais
# je suis partisan de garder ça simple :-)
#
def is_charged(aa, charged_set):
    print(aa in charged_set)
# Ça marche
is_charged("K", "DEKRH")
# Ça marche pas...
is_charged("K")

True


TypeError: is_charged() missing 1 required positional argument: 'charged_set'

## `return` — récupérer une valeur

`print()` affiche quelque chose. `return` remet une valeur à la personne ou au code qui a appelé la fonction.


In [4]:
def is_charged(aa, charged_set):
    return aa in charged_set

result = is_charged("K", "DEKRH")
print(result)

True


`result` contient maintenant `True` — vous pouvez l’utiliser dans un autre `if`, le stocker ou le transmettre. `print()` seul ne le permet pas.


## Un piège courant : appeler avant de définir

Python lit de haut en bas — une fonction doit être définie avant d’être appelée :

```python
greet()              # appelée d’abord — échec

def greet():
    print("Hello, lab!")   # définie ensuite, trop tard
```
```
NameError: name 'greet' is not defined
```

**Correctif :** placez chaque `def` au-dessus du code qui l’appelle. L’ordre des cellules d’un notebook compte aussi — si vous exécutez une cellule qui « appelle la fonction » avant celle qui « définit la fonction », vous obtiendrez la même erreur.


## Une fonction est simplement de la logique emballée

`def nom(paramètres):` l’ouvre. Les lignes indentées forment son corps — ce sont les mêmes `if`, `for` et `in` que vous avez utilisés durant toute la dernière heure. `return` renvoie une valeur. Ensuite : construire le vrai classificateur sous forme de fonction, à partir de la boucle de l’heure 2.


## Construire `classify_aa()`

La logique `if/elif/in` de l’heure 2, emballée dans une fonction.


In [6]:
def classify_aa(aa):
    if aa in "DEKRH":
        return "charged"
    elif aa in "STNQYC":
        return "polar"
    elif aa in "AVLIMFWPG":
        return "hydrophobic"

classify_aa("K")

'charged'

**À essayer :** appelez `classify_aa()` — un résidu entre, une catégorie sort.


In [7]:
classify_aa("W")

'hydrophobic'

## Construire `count_categories()`

Parcourir une séquence entière en appelant `classify_aa()` une fois par résidu.


In [9]:
def count_categories(seq):
    charged, polar, hydrophobic = 0, 0, 0
    for aa in seq:
        cat = classify_aa(aa)
        if cat == "charged":
            charged += 1
        elif cat == "polar":
            polar += 1
        elif cat == "hydrophobic":
            hydrophobic += 1
    return charged, polar, hydrophobic

count_categories("MWKASAGHAV")

(2, 1, 7)

**À essayer :** appelez `count_categories()` — le même résultat que la boucle construite à la main la dernière heure, maintenant réutilisable en une seule ligne.


`(2, 1, 7)` — un tuple : trois valeurs renvoyées ensemble, dans l’ordre : chargés, polaires, hydrophobes.


## Deux petites fonctions, un outil réutilisable

`count_categories("n’importe quelle chaîne de lettres")` — fonctionne maintenant sur n’importe quoi. Vous pourriez l’appeler immédiatement sur les 550 résidus complets de CTTN, tout aussi facilement que sur 10. Mais saisir à la main une protéine de 550 lettres n’est pas réaliste. Les vraies séquences se trouvent dans des fichiers. C’est la prochaine étape.


## Lire un fichier — `open()` et `.readlines()`

Un fichier FASTA sur le disque devient une liste de lignes dans Python.


In [10]:
f = open("../z.misc_files/data_seq/cttn_teaching_multiset.fa")
lines = f.readlines()
f.close()

lines[0]

'>NP_005222.2 src substrate cortactin isoform a [Homo sapiens]\n'

`open()` donne accès au fichier. `.readlines()` le lit sous forme de liste — une chaîne par ligne. `close()` libère le fichier.


## `with open() as` — le modèle plus sûr

Ferme le fichier automatiquement, même si une erreur survient à l’intérieur.


In [ ]:
with open("../z.misc_files/data_seq/cttn_teaching_multiset.fa") as f:
    lines = f.readlines()
# le fichier est déjà fermé ici


Aucun `.close()` n’est nécessaire — le bloc `with` s’en charge. C’est le modèle que vous verrez dans vos notebooks de laboratoire.


## Reconnaître les lignes FASTA

Chaque ligne appartient à l’un de deux types exactement :

| La ligne ressemble à... | C’est... | Test en Python |
|---|---|---|
| `>NP_005222.2 src substrate...` | Un en-tête — il commence un nouvel enregistrement | `line[0] == ">"` |
| `MWKASAGHAVSIAQDDAGADD...` | Des données de séquence — une partie de l’enregistrement courant | `line[0] != ">"` |

`line[0]` — l’indexation d’une chaîne, vue au cours 1 — permet de distinguer les deux types.


**À essayer :** cette ligne est-elle un en-tête ? Une vérification sur une ligne, reprise du modèle `checkFasta()` de Foisy.


In [11]:
line = ">NP_005222.2 src substrate cortactin..."
line[0] == ">"

True

## `.strip()` — retirer le saut de ligne

`readlines()` conserve le `\n` invisible à la fin de chaque ligne.


In [ ]:
lines[1]

In [ ]:
lines[1].strip()

Appliquez toujours `.strip()` à une ligne avant de l’utiliser — un `\n` parasite brisera silencieusement les comparaisons et les concaténations.


## La boîte à outils pour lire un fichier

`with open(path) as f:` → `f.readlines()` → `for line in lines:` → `if line[0] == ">"`, sinon retirer le saut de ligne et recueillir la séquence. C’est tout le modèle. Ensuite : l’assembler et lire les cinq vraies protéines de `cttn_teaching_multiset.fa`.


## Le résultat : cinq vraies protéines

Un fichier. Cinq vraies protéines humaines. Chaque résidu, classé.
`cttn_teaching_multiset.fa` — le même fichier de l’heure Unix du cours 1 :
CTTN, actine, insuline, hémoglobine β, lysozyme C.

Parcourir les lignes → séparer cinq enregistrements → classer chaque résidu de chacun → la réponse littérale à la question de clôture de la semaine dernière : pourrait-on le faire pour 500 gènes ?


## Analyser les cinq enregistrements

Une boucle sur chaque ligne, qui construit une séquence par protéine.


In [12]:
records = {}
name = None
with open("../z.misc_files/data_seq/cttn_teaching_multiset.fa") as f:
    for line in f.readlines():
        if line[0] == ">":
            name = line[1:].split()[0]
            records[name] = ""
        else:
            records[name] += line.strip()

`records` est un dictionnaire : le premier mot de chaque en-tête (l’accession) correspond à la séquence en cours de construction de cette protéine.


**À essayer :** examinez ce que vous avez analysé — cinq clés en entrée, cinq vraies protéines en sortie.


In [13]:
list(records.keys())

['NP_005222.2', 'NP_001092.1', 'NP_000198.1', 'NP_000509.1', 'NP_000230.1']

In [14]:
len(records["NP_005222.2"])

550

550 résidus pour CTTN — ce qui correspond exactement au décompte Unix du cours 1. L’analyseur est correct.


## Classer chaque protéine

Une boucle sur le dictionnaire, appelant `count_categories()` une fois par protéine.


In [ ]:
for accession, seq in records.items():
    charged, polar, hydrophobic = count_categories(seq)
    print(accession, len(seq), charged, polar, hydrophobic)

Trois fonctions construites aujourd’hui, qui travaillent ensemble : analyse du fichier → `count_categories()` → `classify_aa()` pour chaque résidu.

## Le résultat

| Protéine | Longueur | Chargés | Polaires | Hydrophobes |
|---|---:|---:|---:|---:|
| CTTN (NP_005222.2) | 550 | 190 | 145 | 215 |
| Actine (NP_001092.1) | 375 | 95 | 93 | 187 |
| Insuline (NP_000198.1) | 110 | 19 | 28 | 63 |
| Hémoglobine β (NP_000509.1) | 147 | 38 | 26 | 83 |
| Lysozyme C (NP_000230.1) | 148 | 32 | 44 | 72 |

Cinq lignes `print()`. Tout provient de dix lignes de code de fonctions, exécutées une fois par protéine.


## Voici la réponse à la question de la semaine dernière

*« Pourriez-vous faire cela pour 500 gènes ? »* — oui. Le code que vous venez d’exécuter ne se préoccupe pas de savoir si `records` contient 5 entrées ou 500 — la boucle, les fonctions et la logique sont identiques dans les deux cas. C’est cela, le « scripting » : écrire la logique une fois, correctement, puis la taille des données cesse d’être votre problème.


## 📋 Aide-mémoire — fonctions

| Syntaxe | Ce qu’elle fait | Utilisée aujourd’hui comme |
|---|---|---|
| `def name(params):` | Définit un bloc de code réutilisable | `def classify_aa(aa):` |
| `return value` | Renvoie une valeur à l’appelant | `return aa in charged_set` |
| `name(args)` | Appelle une fonction définie | `classify_aa("K")` |
| `a, b = f(x)` | Décompose plusieurs valeurs renvoyées | `charged, polar, h = count_categories(seq)` |

## 📋 Aide-mémoire — fichiers

| Syntaxe | Ce qu’elle fait | Utilisée aujourd’hui comme |
|---|---|---|
| `open(path)` | Ouvre un fichier en lecture | `open("cttn_teaching_multiset.fa")` |
| `f.readlines()` | Lit toutes les lignes dans une liste | `lines = f.readlines()` |
| `with open(...) as f:` | Ouvre et ferme automatiquement un fichier | `with open(path) as f:` |
| `line[0] == ">"` | Détecte une ligne d’en-tête FASTA | `if line[0] == ">":` |
| `.strip()` | Retire le saut de ligne final | `line.strip()` |


## Bilan

Il y a deux heures : variables et chaînes de caractères. Maintenant : fonctions, fichiers et cinq vraies protéines classées.

**Prochaine étape :** TP1b — vous écrirez et exécuterez ce code vous-même. Semaine 3 : les bases de données — là où vivent réellement les données de séquences.
